# AgentMorph Stage 1 - **Gemma-2-9B**

Runs `Gemma-2-9B` on Colab T4. Each model gets a clean Python process so VRAM fragmentation from prior loads doesn't carry over. All 5 per-model notebooks share one `--out-dir` on Drive; the manifest key includes the model id so there are no file collisions.

**Model size:** mid-large (~7 GB)

**Default sweep:** `native + langgraph`. smolagents is intentionally excluded — its 30-tool system prompt blows the T4 context budget by step 3-4 (see §8 note).

**Order:** Llama-3.2-3B (canary) -> Qwen2.5-7B -> Llama-3.1-8B -> Gemma-2-9B -> Phi-4 (last; tightest on T4).

**Before you run:** Runtime -> Change runtime type -> **T4 GPU**. Have your HF token ready for §3.

## 1. GPU sanity check

In [ ]:
import torch
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f"CUDA OK: {torch.cuda.get_device_name(0)} ({dev.total_memory / 1024**3:.1f} GB VRAM)")
else:
    print("NOTE: no CUDA visible. The pipeline will fall back to CPU — "
          "much slower than T4 (~100x) and limited to small models "
          "(Llama-3.2-3B only, likely OOM on 7B+). For production runs "
          "use Runtime > Change runtime type > T4 GPU.")

## 2. Clone / pull the repo

In [ ]:
import os
REPO_URL = "https://github.com/Pranamya16/AgentMorph.git"
REPO_DIR = "/content/AgentMorph"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout main && git pull --ff-only
!cd {REPO_DIR} && git log -1 --oneline

## 3. HuggingFace auth

Paste your token, run the cell, then **clear the token from the cell before sharing this notebook.**

In [ ]:
from huggingface_hub import login
HF_TOKEN = "hf_REPLACE_ME"
login(token=HF_TOKEN)

## 4. Mount Drive + install extras

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -e /content/AgentMorph[models,smolagents,langgraph,agentdojo]

## 5. Set the model for this notebook

Each notebook is named after the model it runs.

In [ ]:
MODEL = "Gemma-2-9B"
print("this notebook will run:", MODEL)

## 6. (Optional) wipe prior `Gemma-2-9B` cells from the shared run

Prunes ONLY this model's rows from the shared `manifest.json` + deletes its JSONL files, leaving other models' progress untouched. Run after an adapter fix when you want to retry this model's trajectories from scratch.

In [ ]:
import json, pathlib
RUN_DIR = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline")
manifest_path = RUN_DIR / "manifest.json"
traj_dir = RUN_DIR / "trajectories"

if manifest_path.exists():
    data = json.loads(manifest_path.read_text())
    before = len(data.get("completed", {}))
    data["completed"] = {k: v for k, v in data.get("completed", {}).items() if not k.startswith(MODEL + "|")}
    after = len(data["completed"])
    manifest_path.write_text(json.dumps(data, indent=2))
    print(f"manifest: {before} -> {after} entries (dropped {before - after} for {MODEL})")
else:
    print("no manifest - nothing to prune")

if traj_dir.exists():
    for p in traj_dir.glob(f"{MODEL}__*.jsonl"):
        p.unlink()
        print("deleted:", p.name)
else:
    print("no trajectory dir yet")

## 7. Smoke test - `Gemma-2-9B` x native x 3 scenarios

Proves the model loads and tools + trajectory logger work end-to-end. Writes to `stage1_smoke_<MODEL>/` (separate from the baseline dir) so it doesn't pollute §8.

In [ ]:
!python -m agentmorph.runner \
    --model {MODEL} --framework native --env ecommerce --n-scenarios 3 \
    --hf-cache-dir /content/drive/MyDrive/AgentMorph/hf_cache \
    --out-dir /content/drive/MyDrive/AgentMorph/runs/stage1_smoke_{MODEL}

## 8. Framework sweep - `Gemma-2-9B` x (native + langgraph) x ecommerce x 20 scenarios

Resumable. Writes into the shared `stage1_baseline/` directory.

**Why `native + langgraph` only (not smolagents):** smolagents injects all 30 tool descriptions into every turn's prompt, which on small open models snowballs input context to 20K+ tokens by step 3 and triggers CUDA OOM on a 14.6 GB T4. The smolagents adapter stays in the code — add `--framework smolagents` below to test it on larger / more tool-calling-capable models (Qwen2.5-7B, Gemma-2-9B, Phi-4).

In [ ]:
import os
# Reduce CUDA allocator fragmentation on long multi-scenario runs.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m agentmorph.runner \
    --model {MODEL} --framework native --framework langgraph --env ecommerce \
    --hf-cache-dir /content/drive/MyDrive/AgentMorph/hf_cache \
    --out-dir /content/drive/MyDrive/AgentMorph/runs/stage1_baseline

## 9. Diagnostic - full error text for Gemma-2-9B

Prints the complete (non-truncated) content of the first error step per framework. Paste back here if anything fails.

In [ ]:
import json, pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/trajectories")
for p in sorted(traj_dir.glob(f"{MODEL}__*.jsonl")):
    print(f"\n======= {p.name} =======")
    shown = False
    for line in p.open():
        t = json.loads(line)
        errs = [s for s in t["steps"] if s["kind"] == "error"]
        if errs and not shown:
            print(f"scenario: {t['scenario_id']}")
            print(f"steps: {len(t['steps'])}  final_answer: {t['final_answer']!r}")
            print("FULL ERROR TEXT:")
            print(errs[0]["content"])
            shown = True
            break
    if not shown:
        print("(no error steps in this file - all trajectories finished cleanly!)")

## 10. Finish-rate for `Gemma-2-9B`

In [ ]:
import json, collections, pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/trajectories")
stats = collections.defaultdict(lambda: [0, 0])
for p in traj_dir.glob(f"{MODEL}__*.jsonl"):
    for line in p.open():
        t = json.loads(line)
        key = (t["model_id"], t["framework_id"])
        stats[key][0] += 1
        if t["final_answer"]:
            stats[key][1] += 1
print(f"=== {MODEL} finish rate ===")
for (m, f), (tot, fin) in sorted(stats.items()):
    pct = 100 * fin / tot if tot else 0
    print(f"  {f:10s}  finished {fin:2d}/{tot:2d}  ({pct:3.0f}%)")

## 11. Peek one trajectory from `Gemma-2-9B`

In [ ]:
import json, pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/trajectories")
for p in sorted(traj_dir.glob(f"{MODEL}__*.jsonl")):
    sample = p.open().readline()
    if not sample:
        continue
    t = json.loads(sample)
    print(f"\n======= {p.name} =======")
    print(f"scenario: {t['scenario_id']}")
    print(f"framework: {t['framework_id']}")
    print(f"steps: {len(t['steps'])}  wall_seconds: {t.get('wall_seconds') or 0:.1f}")
    print(f"final_answer: {t['final_answer']!r}\n")
    for s in t["steps"][:20]:
        tag = s["kind"]
        if tag == "tool_call":
            print(f"  [{tag}] {s['tool_name']}({s['tool_args']})")
        elif tag == "tool_result":
            print(f"  [{tag}] {s['tool_name']} -> {str(s['tool_output'])[:80]}")
        else:
            print(f"  [{tag}] {str(s.get('content', ''))[:200]}")